In [1]:
import pandas as pd
import requests
from xml.etree import ElementTree
import time
from tqdm import tqdm

import os

In [2]:
def fetch_pmids_by_journal(journal_abbrev, start_year=2001, end_year=2024):
    """
    Fetch PMIDs for a given PubMed journal abbreviation.
    Example journal_abbrev: 'J Biomed Inform', 'J Am Soc Nephrol'
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    params = {
        "db": "pubmed",
        "term": f"{journal_abbrev}[Journal]",
        "mindate": start_year,
        "maxdate": end_year,
        "datetype": "pdat",   # publication date
        "retmode": "xml",
        "retmax": 100000
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    root = ElementTree.fromstring(response.content)

    return [id_elem.text for id_elem in root.findall(".//Id")]



def fetch_pubmed_records(pmids, batch_size=200):
    """
    Fetch metadata for a list of PMIDs in batches.
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    all_records = []

    for i in tqdm(range(0, len(pmids), batch_size)):
        batch = pmids[i:i+batch_size]
        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "xml"
        }
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        root = ElementTree.fromstring(response.content)

        for article in root.findall(".//PubmedArticle"):
            record = {
                "pmid": article.findtext(".//PMID"),
                "title": article.findtext(".//ArticleTitle"),
                "abstract": article.findtext(".//Abstract/AbstractText"),
                "year": article.findtext(".//JournalIssue/PubDate/Year"),
                "mesh_terms": [mh.findtext("DescriptorName") for mh in article.findall(".//MeshHeading")],
                "keywords": [kw.text for kw in article.findall(".//KeywordList/Keyword") if kw.text]
            }
            all_records.append(record)

        time.sleep(0.34)  # <= 3 requests/second (NCBI policy)

    return all_records





In [40]:
pmids = fetch_pmids_by_journal("J Am Soc Nephrol", 1990, 2025)
print(f"Found {len(pmids)} JASN articles (1990-2025)")

records = fetch_pubmed_records(pmids, batch_size=200)
print(f"Retrieved {len(records)} records")
print(records[0])  # peek at first record


In [46]:
df = pd.DataFrame(records)
df["mesh_terms"] = df["mesh_terms"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df["keywords"] = df["keywords"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df = df.loc[(df['year']<=2025) & (df['year']>=1990)]
print("Number of records between 1990 and 2025:", len(df))

folder_path = "data/raw"
os.makedirs(folder_path, exist_ok=True)
print(df.shape)
df = df.loc[(df['keywords']!='')&(df['keywords'].notna())]
print(df.shape)
df.year.value_counts().sort_index()
df.to_csv(os.path.join(folder_path, "jasn_pubmed_1990_2025_w_keywords.csv"), index=False)

In [3]:
pmids = fetch_pmids_by_journal("Ann. Rheum. Dis.", 1939, 2025)
print(f"Found {len(pmids)} Annals of Rheumatic Diseases articles (1939-2025)")

records = fetch_pubmed_records(pmids, batch_size=200)
print(f"Retrieved {len(records)} records")
print(records[0])  # peek at first record

In [4]:
df = pd.DataFrame(records)
df["mesh_terms"] = df["mesh_terms"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df["keywords"] = df["keywords"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df = df.loc[(df['year']<=2025) & (df['year']>=1939)]
print("Number of records between 1939 and 2025:", len(df))

folder_path = "data/raw"
os.makedirs(folder_path, exist_ok=True)
print(df.shape)
df = df.loc[(df['keywords']!='')&(df['keywords'].notna())]
print(df.shape)
print(df.year.value_counts().sort_index())
df.to_csv(os.path.join(folder_path, "ard_pubmed_1939_2025_w_keywords.csv"), index=False)